# Faruq-v3 — leaf/ontology gradient-conflict audit

Audit train-only pada checkpoint D0. Membandingkan arah gradien klasifikasi 21 kelas dengan auxiliary ontology tanpa training, validation, atau test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-gradient-conflict-audit-v1/gradient_conflict_audit.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT)

In [ ]:
from coffee_detector.analysis.ontology_gradient_conflict import run_ontology_gradient_conflict_audit
result = run_ontology_gradient_conflict_audit(
    CHECKPOINT, DATA_ROOT, OUTPUT, device='0', batch_size=8, max_batches=24, seed=42
)
assert result['training_executed'] is False
assert result['validation_images_accessed'] is False
assert result['test_images_accessed'] is False
print('AUDIT SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display
rows = [{'parameter_group': group, **summary} for group, summary in result['summaries'].items()]
display(pd.DataFrame(rows))
print('DECISION:', result['decision']['decision'])
print('TRUNK CONFLICT:', result['decision']['feature_extractor_conflict'])
print('HEAD CONFLICT :', result['decision']['classification_head_conflict'])
print('NEXT:', result['decision']['next_action'])
print('TRAINING AUTHORIZED:', result['decision']['training_authorized'])
print('SUMMARY:', result['summary'])
print('Kirim tabel dan keputusan. Jangan training.')